# Lab 02: Data Ingestion — How the Poller Works

In this lab you'll learn how Quant-Sports's poller architecture fetches data from external APIs and stores it in TimescaleDB. By the end you will:

- Understand the poller architecture (sources → writers → DB)
- Read `PollerConfig` and `OddsAPIConfig` / `ESPNInjuriesConfig`
- Query raw odds and injury data
- Aggregate data by book, sport, and team
- Explore TimescaleDB hypertable internals (chunks, compression)
- Manually trigger a poll cycle
- Inspect poller health, run history, and logs

---

## Prerequisites

- **Lab 01 completed** — you know how to connect to TimescaleDB
- A running **TimescaleDB** instance
- The **poller has run at least once** (otherwise some tables will be empty)
- For the manual poll trigger: `QUANT_SPORTS_POLLER_ODDS_API_KEY` set if you want live odds data

### Environment variables for the poller

| Variable | Purpose | Default |
|---|---|---|
| `QUANT_SPORTS_DB_HOST` | Database host | `timescaledb` |
| `QUANT_SPORTS_POLLER_ODDS_API_KEY` | Odds API key | `` (empty = disabled) |
| `QUANT_SPORTS_POLLER_ACTIVE_SPORTS` | Comma-separated sports | `nfl,nba` |

---

## Section 1: Poller Architecture

Quant-Sports's data pipeline follows a simple, robust pattern:

```
┌─────────────┐     ┌──────────────┐     ┌──────────────┐     ┌──────────┐
│  Odds API   │────▶│  parse_odds   │────▶│ write_odds   │────▶│ odds_ticks│
│  (external) │     │  _api_to_ticks│     │ _ticks (COPY)│     │ (hyper)  │
└─────────────┘     └──────────────┘     └──────────────┘     └──────────┘

┌─────────────┐     ┌──────────────┐     ┌──────────────┐     ┌──────────┐
│  ESPN Inj.  │────▶│  parse_espn  │────▶│ write_inj.   │────▶│ injuries │
│  (external) │     │  _to_rows     │     │ (COPY)       │     │ (hyper)  │
└─────────────┘     └──────────────┘     └──────────────┘     └──────────┘
```

Key design decisions:

- **Async everywhere** — all fetch and write operations are `async def`
- **COPY protocol** — bulk inserts use PostgreSQL's fastest path
- **Idempotent** — the poller can be re-run safely
- **Per-source, per-sport tasks** — `run_odds_api_poll(config, 'nfl')` is independent
  from `run_espn_poll(config, 'nba')`
- **Health tracking** — every cycle writes to `poller_runs` and `poller_health`

In [ ]:
# Cell 4: Imports and connection setup
import asyncio
import json

from quantitative_sports.infra.db.connection import DBConfig, DatabasePool, get_pool, health_check, reset_pool
from quantitative_sports.infra.db.queries import (
    get_poller_health_summary,
    get_poller_runs,
    get_poller_logs,
    get_table_stats,
    get_db_size,
    get_poller_success_rates,
    get_recent_poll_volume,
    refresh_db_metrics,
)
from quantitative_sports.infra.db.schema import verify_schema, EXPECTED_TABLES
from quantitative_sports.infra.poller.config import PollerConfig, get_active_sports, is_odds_api_enabled, is_espn_enabled
from quantitative_sports.infra.poller.sources.odds_api import OddsAPIConfig
from quantitative_sports.infra.poller.sources.espn_injuries import ESPNInjuriesConfig, SPORT_ENDPOINTS
from quantitative_sports.infra.poller.tasks import run_poll_cycle, TaskResult

import nest_asyncio
nest_asyncio.apply()

# Connect to the database
config = DBConfig.from_env()
pool = await get_pool(config)
print(f"Connected to {config.host}:{config.port}/{config.database}")

---

## Section 2: Poller Configuration

The `PollerConfig` aggregates all sub-configs. Let's inspect it to understand what's configurable.

In [ ]:
# Cell 6: Inspect PollerConfig defaults
poller_config = PollerConfig()
print(f"Scheduler: {poller_config.scheduler}")
print(f"Active sports: {poller_config.active_sports}  →  parsed: {get_active_sports(poller_config)}")
print(f"Max retries: {poller_config.max_retries}")
print(f"Retry delay: {poller_config.retry_delay}s")
print(f"Log level: {poller_config.log_level}")
print(f"Prefect API URL: {poller_config.prefect_api_url}")

In [ ]:
# Cell 7: Inspect source sub-configs
print("Odds API Config:")
print(f"  base_url: {poller_config.odds_api.base_url}")
print(f"  regions: {poller_config.odds_api.regions}")
print(f"  markets: {poller_config.odds_api.markets}")
print(f"  odds_format: {poller_config.odds_api.odds_format}")
print(f"  interval_seconds: {poller_config.odds_api.interval_seconds}")
print(f"  API key set: {'Yes' if poller_config.odds_api.api_key else 'No (disabled)'}")
print(f"  Enabled: {is_odds_api_enabled(poller_config)}")

print("\nESPN Injuries Config:")
print(f"  base_url: {poller_config.espn.base_url}")
print(f"  interval_seconds: {poller_config.espn.interval_seconds}s ({poller_config.espn.interval_seconds // 60} min)")
print(f"  user_agent: {poller_config.espn.user_agent}")
print(f"  Supported sports: {', '.join(sorted(SPORT_ENDPOINTS.keys()))}")
print(f"  Enabled: {is_espn_enabled(poller_config)}")

In [ ]:
# Cell 8: DBConfig details
print("Database Config:")
print(f"  host: {poller_config.db.host}")
print(f"  port: {poller_config.db.port}")
print(f"  user: {poller_config.db.user}")
print(f"  database: {poller_config.db.database}")
print(f"  DSN: {poller_config.db.to_dsn().split('@')[1]}")  # Hide password

---

## Section 3: Raw Odds Data

The `odds_ticks` hypertable is where all odds data lands. Each row represents a single price point for a selection at a bookmaker.

Columns: `sport`, `league`, `event_id`, `book`, `market`, `selection`, `price`, `line`, `ts`, `source_raw`.

In [ ]:
# Cell 10: Latest 20 odds ticks
odds_rows = await pool.fetch(
    "SELECT sport, league, event_id, book, market, selection, price, line, ts "
    "FROM odds_ticks ORDER BY ts DESC LIMIT 20"
)

if odds_rows:
    print(f"Found {len(odds_rows)} recent odds ticks:\n")
    for row in odds_rows[:10]:
        print(f"  {row['ts']} | {row['sport']}/{row['market']} | {row['book']} {row['selection']} @ {row['price']}")
    if len(odds_rows) > 10:
        print(f"  ... and {len(odds_rows) - 10} more")
else:
    print("No odds_ticks data yet. The poller hasn't run, or no data was fetched.")

In [ ]:
# Cell 11: Odds ticks schema details
cols = await pool.fetch(
    "SELECT column_name, data_type, is_nullable "
    "FROM information_schema.columns "
    "WHERE table_schema = 'public' AND table_name = 'odds_ticks' "
    "ORDER BY ordinal_position"
)
print("odds_ticks columns:")
for c in cols:
    print(f"  {c['column_name']:<15} {c['data_type']:<25} nullable={c['is_nullable']}")

---

## Section 4: Aggregate Odds by Book and Sport

Understanding which books and sports have the most data helps you identify coverage gaps and plan analyses.

In [ ]:
# Cell 13: Aggregate odds_ticks by book and sport
agg = await pool.fetch(
    "SELECT book, sport, COUNT(*) AS cnt, "
    "MIN(ts) AS earliest, MAX(ts) AS latest "
    "FROM odds_ticks GROUP BY book, sport ORDER BY cnt DESC"
)

if agg:
    print(f"{'Book':<20} {'Sport':<10} {'Count':>10} {'Earliest':<25} {'Latest':<25}")
    print(f"{'-'*20} {'-'*10} {'-'*10} {'-'*25} {'-'*25}")
    for row in agg[:15]:
        print(f"{row['book']:<20} {row['sport']:<10} {row['cnt']:>10} {str(row['earliest']):<25} {str(row['latest']):<25}")
    if len(agg) > 15:
        print(f"\n... and {len(agg) - 15} more book/sport combinations")
else:
    print("No odds data to aggregate. Run the poller first.")

In [ ]:
# Cell 14: Breakdown by market type
markets = await pool.fetch(
    "SELECT market, COUNT(*) AS cnt FROM odds_ticks GROUP BY market ORDER BY cnt DESC"
)
if markets:
    print("Market distribution:")
    for row in markets:
        print(f"  {row['market']:<15} {row['cnt']:>10} ticks")
else:
    print("No data yet.")

---

## Section 5: Raw Injury Data

The `injuries` hypertable stores ESPN injury reports. Each row represents a single player injury status at a point in time.

Columns: `sport`, `player_id`, `player_name`, `team`, `position`, `status`, `detail`, `ts`, `source_raw`.

In [ ]:
# Cell 16: Latest 20 injury reports
injury_rows = await pool.fetch(
    "SELECT sport, player_name, team, position, status, detail, ts "
    "FROM injuries ORDER BY ts DESC LIMIT 20"
)

if injury_rows:
    print(f"Found {len(injury_rows)} recent injury reports:\n")
    for row in injury_rows[:10]:
        detail_str = row['detail'] or 'N/A'
        print(f"  {row['ts']} | {row['sport']}/{row['team']} | {row['player_name']} ({row['status']}) — {detail_str}")
    if len(injury_rows) > 10:
        print(f"  ... and {len(injury_rows) - 10} more")
else:
    print("No injury data yet. The ESPN poller hasn't run, or no data was fetched.")

---

## Section 6: Aggregate Injuries by Team and Status

Understanding injury patterns by team helps identify which teams have the most health concerns.

In [ ]:
# Cell 18: Injuries by team and status
team_injuries = await pool.fetch(
    "SELECT team, status, COUNT(*) AS cnt "
    "FROM injuries GROUP BY team, status ORDER BY cnt DESC"
)

if team_injuries:
    print(f"{'Team':<10} {'Status':<15} {'Count':>10}")
    print(f"{'-'*10} {'-'*15} {'-'*10}")
    for row in team_injuries[:20]:
        print(f"{row['team']:<10} {row['status']:<15} {row['cnt']:>10}")
    if len(team_injuries) > 20:
        print(f"\n... and {len(team_injuries) - 20} more")
else:
    print("No injury data to aggregate.")

In [ ]:
# Cell 19: Injury count by sport
sport_injuries = await pool.fetch(
    "SELECT sport, COUNT(*) AS cnt FROM injuries GROUP BY sport ORDER BY cnt DESC"
)
if sport_injuries:
    print("Injury reports per sport:")
    for row in sport_injuries:
        print(f"  {row['sport']:<10} {row['cnt']:>10} reports")
else:
    print("No data.")

---

## Section 7: Hypertable Internals

TimescaleDB partitions hypertables into **chunks** — smaller, more manageable slices of data. Each chunk covers a time interval (1 day by default in Quant-Sports).

Let's inspect how data is partitioned.

In [ ]:
# Cell 21: Inspect hypertable chunks
try:
    chunks = await pool.fetch(
        "SELECT hypertable_name, chunk_name, range_start, range_end, "
        "is_compressed FROM timescaledb_information.chunks "
        "ORDER BY hypertable_name, range_start"
    )
    if chunks:
        print(f"Found {len(chunks)} chunks:\n")
        print(f"{'Hypertable':<20} {'Chunk':<30} {'Range Start':<25} {'Compressed':<12}")
        print(f"{'-'*20} {'-'*30} {'-'*25} {'-'*12}")
        for c in chunks[:15]:
            print(f"{c['hypertable_name']:<20} {c['chunk_name']:<30} {str(c['range_start']):<25} {str(c['is_compressed']):<12}")
        if len(chunks) > 15:
            print(f"\n... and {len(chunks) - 15} more chunks")
    else:
        print("No chunks yet. Data will be partitioned into chunks as it accumulates.")
except Exception as e:
    print(f"Could not query chunks: {e}")

In [ ]:
# Cell 22: Chunk row counts (approximate)
# Each chunk is a regular Postgres table under the hood
try:
    chunk_stats = await pool.fetch(
        "SELECT hypertable_name, COUNT(*) AS num_chunks, "
        "MIN(range_start) AS earliest_chunk, MAX(range_end) AS latest_chunk "
        "FROM timescaledb_information.chunks "
        "GROUP BY hypertable_name"
    )
    if chunk_stats:
        print("Chunk summary per hypertable:")
        for cs in chunk_stats:
            print(f"  {cs['hypertable_name']}: {cs['num_chunks']} chunks, "
                  f"spanning {cs['earliest_chunk']} → {cs['latest_chunk']}")
    else:
        print("No chunks yet.")
except Exception as e:
    print(f"Could not query chunk stats: {e}")

---

## Section 8: Compression and Retention Policies

TimescaleDB supports automatic compression and retention policies. Quant-Sports configures:

- **Compression**: After 7 days, chunks are compressed to save disk space
- **Retention**: Data older than 2 years is automatically dropped

These policies are set up in the schema DDL.

In [ ]:
# Cell 24: Check compression policies
try:
    comp_policies = await pool.fetch(
        "SELECT hypertable_name, schedule_interval, config "
        FROM timescaledb_information.jobs "
        WHERE proc_name = 'policy_compression' "
        ORDER BY hypertable_name"
    )
    if comp_policies:
        print("Compression policies:")
        for p in comp_policies:
            print(f"  {p['hypertable_name']}: schedule={p['schedule_interval']}")
    else:
        print("No compression policies found (or TimescaleDB community edition).")
except Exception as e:
    print(f"Could not query compression policies: {e}")

In [ ]:
# Cell 25: Check retention policies
try:
    ret_policies = await pool.fetch(
        "SELECT hypertable_name, schedule_interval, config "
        FROM timescaledb_information.jobs "
        WHERE proc_name = 'policy_retention' "
        ORDER BY hypertable_name"
    )
    if ret_policies:
        print("Retention policies:")
        for p in ret_policies:
            print(f"  {p['hypertable_name']}: schedule={p['schedule_interval']}")
    else:
        print("No retention policies found.")
except Exception as e:
    print(f"Could not query retention policies: {e}")

---

## Section 9: Manually Trigger a Poll Cycle

The `run_poll_cycle()` function runs all enabled pollers once. This is useful for:

- Testing the pipeline end-to-end
- Populating fresh data during development
- Debugging source connectivity

**Note**: If the Odds API key is not set, the odds poller will be skipped. The ESPN source requires no API key.

In [ ]:
# Cell 27: Count rows before the poll cycle
odds_count_before = await pool.fetchval("SELECT count(*) FROM odds_ticks")
injury_count_before = await pool.fetchval("SELECT count(*) FROM injuries")
print(f"Before poll: {odds_count_before} odds_ticks, {injury_count_before} injuries")

In [ ]:
# Cell 28: Run a single poll cycle
#
# This will:
#   1. Fetch odds from the Odds API (if key is set) for each active sport
#   2. Fetch injuries from ESPN for each active sport
#   3. Write all data to the database
#   4. Update poller_runs and poller_health
#   5. Run a health check
#
# ⚠️  This may take 10-30 seconds depending on network conditions.

results: list[TaskResult] = await run_poll_cycle(poller_config)

for r in results:
    status_icon = "✅" if r.status == "success" else "❌"
    print(f"{status_icon} {r.poller_name}: {r.status} ({r.rows_written} rows, {r.duration_seconds:.1f}s)")
    if r.error:
        print(f"   Error: {r.error[:200]}")

In [ ]:
# Cell 29: Count rows after the poll cycle
odds_count_after = await pool.fetchval("SELECT count(*) FROM odds_ticks")
injury_count_after = await pool.fetchval("SELECT count(*) FROM injuries")
print(f"After poll:  {odds_count_after} odds_ticks (+{odds_count_after - odds_count_before}), "
      f"{injury_count_after} injuries (+{injury_count_after - injury_count_before})")

---

## Section 10: Poller Health and Run History

Quant-Sports tracks every poll cycle in `poller_runs` and maintains a health dashboard in `poller_health`. The read-side query helpers give you structured access to this data.

In [ ]:
# Cell 31: Poller health summary
health_summary = await get_poller_health_summary(pool)
if health_summary:
    print(f"{'Poller':<30} {'Status':<12} {'Failures':<10} {'Last Run':<12} {'Rows':<8}")
    print(f"{'-'*30} {'-'*12} {'-'*10} {'-'*12} {'-'*8}")
    for entry in health_summary:
        print(f"{entry['poller_name']:<30} {entry['status']:<12} "
              f"{entry['consecutive_failures']:<10} {entry.get('last_run_status', 'N/A'):<12} "
              f"{entry.get('last_run_rows', 'N/A'):<8}")
else:
    print("No poller health entries. The poller hasn't run yet.")

In [ ]:
# Cell 32: Recent poller runs
#
# Let's look at the run history for the ESPN NFL poller
# (or whichever poller has data)

# First, find which pollers have runs
active_pollers = await pool.fetch(
    "SELECT DISTINCT poller_name FROM poller_runs ORDER BY poller_name"
)
poller_names = [row['poller_name'] for row in active_pollers]
print(f"Active pollers with run history: {poller_names}")

# Pick the first one that exists
if poller_names:
    target = poller_names[0]
    runs = await get_poller_runs(pool, target, limit=10)
    print(f"\nLast 10 runs for {target}:")
    for run in runs:
        status_icon = "✅" if run['status'] == 'success' else "❌"
        print(f"  {status_icon} {run['started_at']} → {run['status']} ({run['rows_written']} rows, {run.get('error', '')[:80]})")
else:
    print("No poller_runs data yet.")

In [ ]:
# Cell 33: Poller success rates (last 24 hours)
success_rates = await get_poller_success_rates(pool, hours=24)
if success_rates:
    print(f"{'Poller':<30} {'Runs':>6} {'Success':>8} {'Failed':>8} {'Rate':>8} {'Avg Duration':>12}")
    print(f"{'-'*30} {'-'*6} {'-'*8} {'-'*8} {'-'*8} {'-'*12}")
    for sr in success_rates:
        avg_dur = sr.get('avg_duration_seconds')
        dur_str = f"{avg_dur:.1f}s" if avg_dur else "N/A"
        print(f"{sr['poller_name']:<30} {sr['total_runs']:>6} {sr['successful_runs']:>8} "
              f"{sr['failed_runs']:>8} {sr['success_rate']:>8.1%} {dur_str:>12}")
else:
    print("No poller_runs data in the last 24 hours.")

---

## Section 11: Poller Logs

Every poll cycle writes structured log entries to `poller_logs`. Use `get_poller_logs()` to tail recent log lines for a specific poller.

In [ ]:
# Cell 35: Get recent poller logs
if poller_names:
    target = poller_names[0]
    logs = await get_poller_logs(pool, target, limit=20)
    print(f"Last 20 log lines for {target}:")
    for log_entry in logs:
        level = log_entry.get('level', 'INFO')
        msg = log_entry.get('message', '')[:120]
        print(f"  [{level}] {msg}")
else:
    print("No poller_logs data yet.")

In [ ]:
# Cell 36: Recent poll volume (rows written per source per hour)
volume = await get_recent_poll_volume(pool, hours=24)
if volume:
    print(f"{'Hour Bucket':<25} {'Source':<15} {'Rows':>10}")
    print(f"{'-'*25} {'-'*15} {'-'*10}")
    for v in volume[:20]:
        print(f"{v['bucket']:<25} {v['source']:<15} {v['total_rows']:>10}")
    if len(volume) > 20:
        print(f"\n... and {len(volume) - 20} more entries")
else:
    print("No poll volume data in the last 24 hours.")

---

## Exercises

Try these on your own:

1. **Find the longest-running poller** — Query `poller_runs` to find which poller has the highest average `finished_at - started_at` duration:
   ```sql
   SELECT poller_name,
          AVG(EXTRACT(EPOCH FROM (finished_at - started_at))) AS avg_seconds
   FROM poller_runs
   WHERE status = 'success'
   GROUP BY poller_name
   ORDER BY avg_seconds DESC;
   ```

2. **Which book has the most NFL data?** — Find the bookmaker with the most `odds_ticks` rows for `sport = 'americanfootball_nfl'`.

3. **Injury report timeline** — For a specific team, show the timeline of injury status changes:
   ```sql
   SELECT player_name, status, ts
   FROM injuries
   WHERE team = 'KC'
   ORDER BY ts DESC;
   ```

4. **Compression status** — Check how many chunks are already compressed:
   ```sql
   SELECT hypertable_name,
          COUNT(*) AS total_chunks,
          COUNT(*) FILTER (WHERE is_compressed) AS compressed_chunks
   FROM timescaledb_information.chunks
   GROUP BY hypertable_name;
   ```

---

## Summary

In this lab you learned:

- The poller architecture: sources → parsers → writers → TimescaleDB
- How `PollerConfig` controls scheduler, sports, and intervals
- How to query raw `odds_ticks` and `injuries` data
- Aggregation queries by book, sport, team, and status
- TimescaleDB hypertable internals: chunks, compression, retention
- How to manually trigger a poll cycle with `run_poll_cycle`
- How to inspect poller health, runs, success rates, and logs

### Key Takeaways

| Concept | Function | Table |
|---|---|---|
| Fetch odds | `run_odds_api_poll(config, sport)` | `odds_ticks` |
| Fetch injuries | `run_espn_poll(config, sport)` | `injuries` |
| Run all pollers | `run_poll_cycle(config)` | `poller_runs` + `poller_health` |
| Health dashboard | `get_poller_health_summary(pool)` | `poller_health` |
| Run history | `get_poller_runs(pool, name)` | `poller_runs` |
| Log tail | `get_poller_logs(pool, name)` | `poller_logs` |

### Next Steps

Continue to **Lab 03: Expected Value Calculation** to learn how to convert raw odds into implied probabilities and find +EV bets.

---

*Don't forget to close the pool:*
```python
await pool.close()
```

In [ ]:
# Cell 39: Close the connection pool
await pool.close()
print("Connection pool closed. Lab 02 complete!")